# Every Point Gets Its Own Bandwidth

**Topic:** t-SNE perplexity &nbsp;·&nbsp; **Difficulty:** Hard

t-SNE does not use one bandwidth for the whole dataset. It gives every point its
own `sigma`, chosen so that the entropy of that point's neighbour distribution
matches a target you set (the perplexity). In a dense region `sigma` comes out
small, and in a sparse one it comes out large.

Implement the binary search that finds those sigmas. It is the part of t-SNE that
actually adapts to the data.

## The data (created in the next cell)

- `X` `(300, 5)`: two clusters with very different densities.
- `target_perplexity = 30.0`.

## Your task

With **NumPy only**:

1. For each point `i`, define `p_{j|i} = exp(-d_ij^2 / (2 sigma_i^2))` over
   `j != i`, normalized to sum to 1.
2. The perplexity of that distribution is `2^H` where `H` is its Shannon entropy
   in bits.
3. Binary search `sigma_i` (start with the bracket `[1e-6, 1e6]`, 60 iterations)
   so that the perplexity equals the target to within `1e-5`.
4. Set `answers` to a dict with `"sigmas"` (length-300), `"perplexities"`
   (length-300, the achieved values), `"sigma_dense_mean"` and
   `"sigma_sparse_mean"`: the mean sigma over the first 150 points (dense
   cluster) and the last 150 (sparse).

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks every achieved perplexity is within `1e-4` of the target and that the
sparse cluster's mean sigma is more than twice the dense cluster's.

In [ ]:
import numpy as np

rng = np.random.default_rng(3010)
dense = rng.normal(0, 0.3, (150, 5))
sparse = rng.normal(6.0, 2.5, (150, 5))
X = np.vstack([dense, sparse])
target_perplexity = 30.0
print(f"X {X.shape}: 150 tight points (std 0.3) and 150 loose ones (std 2.5)")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("sigmas", [300], 345.692527043812, 29.395460223839343, 1e-06),
    ("perplexities", [300], 9000.000000000902, 412.9694519381564, 1e-06),
    ("sigma_dense_mean", [], 0.24540591278918308, -0.04109027886116093, 1e-06),
    ("sigma_sparse_mean", [], 2.0592109341695632, -0.34479019090166196, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "perplexities" in answers:
    _p = np.asarray(answers["perplexities"], dtype=float)
    _hit = float(np.abs(_p - target_perplexity).max()) < 1e-4
    _adapt = answers["sigma_sparse_mean"] > 2 * answers["sigma_dense_mean"]
    print(f"  {'perplexity matched':<24} {'ok' if _hit else 'wrong'}  "
          f"(max error {np.abs(_p - target_perplexity).max():.2e})")
    print(f"  {'sigma adapts to density':<24} {'ok' if _adapt else 'wrong'}  "
          f"({answers['sigma_dense_mean']:.3f} vs {answers['sigma_sparse_mean']:.3f})")
    _ok = _ok and _hit and _adapt

print("=" * 56)
if _ok:
    print("PASSED, nice work!")
else:
    print("NOT PASSED yet, revisit your solution and try again.")

## Solution

Perplexity is monotone increasing in `sigma`. A larger bandwidth spreads the
neighbour distribution and raises its entropy, so a plain bisection converges
reliably. The adaptive bandwidth is why t-SNE handles clusters of different
density at all: with one global `sigma`, the dense cluster's neighbours would all
have essentially equal probability and the sparse cluster's distribution would
collapse onto its single nearest point.

```python
import numpy as np

D2 = ((X[:, None, :] - X[None, :, :]) ** 2).sum(axis=2)
n = len(X)

def perp_for(i, sigma):
    d = np.delete(D2[i], i)
    logits = -d / (2 * sigma ** 2)
    logits -= logits.max()
    p = np.exp(logits)
    s = p.sum()
    if s <= 0:
        return 1.0
    p /= s
    nz = p[p > 0]
    H = -np.sum(nz * np.log2(nz))
    return float(2 ** H)

sigmas = np.zeros(n)
perplexities = np.zeros(n)
for i in range(n):
    lo, hi = 1e-6, 1e6
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        if perp_for(i, mid) < target_perplexity:
            lo = mid
        else:
            hi = mid
    sigmas[i] = 0.5 * (lo + hi)
    perplexities[i] = perp_for(i, sigmas[i])

answers = {
    "sigmas": sigmas,
    "perplexities": perplexities,
    "sigma_dense_mean": float(sigmas[:150].mean()),
    "sigma_sparse_mean": float(sigmas[150:].mean()),
}
```